# EPCOT World Showcase: three rankings that disagree

Eleven country pavilions, ranked three ways: how much there is to do, food
and drink, and how authentic each one is.

There is no overall score in this dataset, deliberately. Combining the three
would need weights, and any weighting is a choice presented as arithmetic.
This notebook loads the rankings, draws the disagreement between them, and
checks whether any of it is statistically distinguishable from noise.

Spoiler on that last point: it is not, and the dataset says so up front.
Eleven pavilions is eleven data points.

In [ ]:
import os
import csv
import glob
import math
import matplotlib.pyplot as plt

# Find the data wherever it is mounted. Kaggle puts attached inputs under
# /kaggle/input, but the folder it uses is not always the dataset slug and
# the files are not always at the top of it, so this walks the tree instead
# of assuming a path.
PATTERNS = ['/kaggle/input/**/rankings.csv', '../input/**/rankings.csv',
            'data/rankings.csv', 'rankings.csv', '../data/rankings.csv',
            '**/rankings.csv']

def find_base():
    for pattern in PATTERNS:
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return os.path.dirname(hits[0]) or '.'
    tree = sorted(glob.glob('/kaggle/input/**', recursive=True))[:40]
    raise FileNotFoundError(
        'Could not find rankings.csv.\n\n'
        'On Kaggle: click Add Input at the top right and attach the '
        'EPCOT World Showcase dataset. Importing a notebook file does not '
        'attach it for you.\n\n'
        'What is actually under /kaggle/input:\n  %s\n\n'
        'Locally: run this from the repository root, where the CSVs are in '
        'data/.' % ('\n  '.join(tree) if tree else '(nothing attached)'))

BASE = find_base()
print('reading from %s' % BASE)

def read(name):
    with open(os.path.join(BASE, name), newline='', encoding='utf-8') as fh:
        return list(csv.DictReader(fh))

rank = read('rankings.csv')
print('%d pavilions, %d columns' % (len(rank), len(rank[0])))

## The matrix

One row per pavilion, one column per question. Lower is better.

`authenticity_rank` is blank for The American Adventure. Coding menu items as
native to the cuisine is meaningless when the country is the United States and
the baseline 'generic theme park food' is that country's food, so it is
excluded from that family and still ranks on the other two.

In [ ]:
rows = sorted(rank, key=lambda r: float(r['things_to_do_rank']))

print('%-24s %8s %8s %12s' % ('pavilion', 'things', 'food', 'authentic'))
print('-' * 56)
for r in rows:
    a = r['authenticity_rank']
    print('%-24s %8.1f %8.1f %12s'
          % (r['pavilion'], float(r['things_to_do_rank']),
             float(r['food_rank']), ('%.1f' % float(a)) if a else 'n/a'))

## Where the three disagree

Each line is a pavilion, tracked across the three rankings. A flat line means
the three questions agree about that pavilion. A line that crosses the chart
means they do not.

Three are highlighted because they are the shape of the finding. Everything
else is grey, because eleven coloured lines would be a plate of spaghetti and
no categorical palette survives eleven slots.

In [ ]:
# Palette carried from the rest of this project and checked with a
# colourblindness validator: worst adjacent pair separates at dE 21.4 under
# protanopia, 26.4 for normal vision, all three clear 3:1 against the surface.
BLUE, RED, PURPLE = '#2a78d6', '#e34948', '#7a4fbf'
GREY, INK, INK2, RULE = '#898781', '#0b0b0b', '#52514e', '#c3c2b7'
SURFACE = '#fcfcfb'

HIGHLIGHT = {
    'France': (BLUE,   'good at all three'),
    'Norway': (RED,    'last on all three'),
    'Japan':  (PURPLE, 'ninth in things to do, first in authenticity'),
}

STAGES = [('things_to_do_rank', 'Things to do'),
          ('food_rank', 'Food and drink'),
          ('authenticity_rank', 'Authenticity')]

fig, ax = plt.subplots(figsize=(10, 7.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

for r in rank:
    xs, ys = [], []
    for i, (key, _) in enumerate(STAGES):
        if r[key]:
            xs.append(i)
            ys.append(float(r[key]))
    colour, _ = HIGHLIGHT.get(r['pavilion'], (GREY, None))
    hot = r['pavilion'] in HIGHLIGHT
    ax.plot(xs, ys, color=colour, linewidth=2.4 if hot else 1.4,
            alpha=1.0 if hot else 0.45, zorder=3 if hot else 2,
            marker='o', markersize=7 if hot else 5,
            markerfacecolor=colour, markeredgecolor=SURFACE,
            markeredgewidth=2)
    # Direct labels, so identity never rests on colour alone.
    # A pavilion with no authenticity rank ends mid chart, so its label
    # would sit on top of other lines. The surface-coloured box keeps every
    # label readable without moving any of them.
    ax.text(xs[-1] + 0.06, ys[-1], r['pavilion'],
            va='center', ha='left', fontsize=11 if hot else 9.5,
            color=INK if hot else INK2, zorder=5,
            fontweight='bold' if hot else 'normal',
            bbox=dict(facecolor=SURFACE, edgecolor='none',
                      boxstyle='round,pad=0.18'))

ax.set_xticks(range(len(STAGES)))
ax.set_xticklabels([s[1] for s in STAGES], fontsize=12, color=INK2)
ax.set_xlim(-0.35, 2.95)
ax.set_ylim(11.6, 0.4)
ax.set_yticks(range(1, 12))
ax.set_ylabel('rank, 1 is best', fontsize=11, color=INK2)
ax.tick_params(length=0, colors=INK2)
ax.grid(axis='y', color=RULE, linewidth=0.6, alpha=0.5)
ax.set_axisbelow(True)
for s in ('top', 'right', 'bottom', 'left'):
    ax.spines[s].set_visible(False)
ax.set_title('The same eleven pavilions, asked three different questions',
             fontsize=15, color=INK, pad=16, loc='left')

handles = [plt.Line2D([], [], color=c, linewidth=2.4, marker='o',
                      markersize=7, label='%s, %s' % (n, why))
           for n, (c, why) in HIGHLIGHT.items()]
handles.append(plt.Line2D([], [], color=GREY, linewidth=1.4, alpha=0.45,
                          marker='o', markersize=5, label='the other eight'))
ax.legend(handles=handles, loc='lower left', bbox_to_anchor=(0, -0.22),
          frameon=False, ncol=2, fontsize=10, labelcolor=INK2)

plt.tight_layout()
plt.show()

## Is any of this a real relationship?

Spearman's rho between each pair of rankings, against the critical value for
this sample size. The critical value is the thing most rank comparisons leave
out, and it is doing all the work here.

In [ ]:
def spearman(a, b):
    pairs = [(x, y) for x, y in zip(a, b) if x is not None and y is not None]
    n = len(pairs)
    d2 = sum((x - y) ** 2 for x, y in pairs)
    return 1 - (6 * d2) / (n * (n * n - 1)), n

def col(key):
    return [float(r[key]) if r[key] else None for r in rank]

# Two sided critical values for Spearman's rho at p = 0.05.
CRIT = {10: 0.648, 11: 0.618}

print('%-34s %7s %4s %9s   %s'
      % ('comparison', 'rho', 'n', 'needs', 'verdict'))
print('-' * 74)
for ka, kb, label in (
        ('things_to_do_rank', 'authenticity_rank', 'things to do vs authenticity'),
        ('things_to_do_rank', 'food_rank',         'things to do vs food'),
        ('food_rank',         'authenticity_rank', 'food vs authenticity')):
    rho, n = spearman(col(ka), col(kb))
    need = CRIT[n]
    ok = abs(rho) >= need
    print('%-34s %+7.3f %4d %9.2f   %s'
          % (label, rho, n, need,
             'significant' if ok else 'NOT distinguishable from zero'))

print()
print('None of the three clears the bar. The honest reading is that the')
print('questions have unrelated answers, not that there is a weak positive')
print('relationship. Reporting +0.25 as a trend on eleven points would be')
print('the same overreach as a ratio computed from one observation.')

## Things to do against authenticity, as a field

The same two rankings as a scatter. If there were a tradeoff between how much
a pavilion offers and how real it feels, the points would fall along a line.
They do not. They fill the space, which is what 'unrelated' looks like.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

mid = 6
ax.axvline(mid, color=RULE, linewidth=1)
ax.axhline(mid, color=RULE, linewidth=1)
for x, y, t in ((3.2, 3.2, 'plenty to do\nand it feels real'),
                (8.8, 3.2, 'feels real,\nnot much to do'),
                (3.2, 8.8, 'plenty to do,\nfeels thinner'),
                (8.8, 8.8, 'neither')):
    ax.text(x, y, t, fontsize=10, color=RULE, ha='center', va='center',
            zorder=1)

for r in rank:
    if not r['authenticity_rank']:
        continue
    x = float(r['things_to_do_rank'])
    y = float(r['authenticity_rank'])
    colour, _ = HIGHLIGHT.get(r['pavilion'], (GREY, None))
    hot = r['pavilion'] in HIGHLIGHT
    ax.scatter([x], [y], s=190 if hot else 110, color=colour,
               edgecolor=SURFACE, linewidth=2, zorder=4 if hot else 3,
               alpha=1.0 if hot else 0.6)
    ax.annotate(r['pavilion'], (x, y), textcoords='offset points',
                xytext=(0, 15), ha='center', fontsize=10.5 if hot else 9.5,
                color=INK if hot else INK2,
                fontweight='bold' if hot else 'normal')

ax.set_xlim(0.2, 11.8)
ax.set_ylim(11.8, 0.2)
ax.set_xlabel('things to do rank, 1 is most', fontsize=11, color=INK2)
ax.set_ylabel('authenticity rank, 1 is most', fontsize=11, color=INK2)
ax.set_xticks(range(1, 12))
ax.set_yticks(range(1, 12))
ax.tick_params(length=0, colors=INK2)
ax.grid(color=RULE, linewidth=0.5, alpha=0.35)
ax.set_axisbelow(True)
for s in ('top', 'right', 'bottom', 'left'):
    ax.spines[s].set_visible(False)
ax.set_title('No tradeoff, just uneven investment',
             fontsize=15, color=INK, pad=16, loc='left')
plt.tight_layout()
fig.subplots_adjust(bottom=0.12)
fig.text(0.08, 0.025, 'The American Adventure is absent: it is excluded '
         'from the authenticity ranking.', fontsize=9.5, color=INK2)
plt.show()

## Underneath the authenticity ranking: the menus

`dining.csv` carries all 1,513 published menu items, each coded for whether
the dish belongs to that country's cuisine. This is the single biggest input
to the authenticity ranking and the one with the most judgment in it, so it is
worth looking at directly rather than taking the rank on trust.

The coding rules and every borderline call are written out in the repository.

In [ ]:
import collections

dining = read('dining.csv')
tally = collections.defaultdict(collections.Counter)
for r in dining:
    if r['cuisine_code'] and r['status'] == 'open':
        tally[r['pavilion']][r['cuisine_code']] += 1

order = sorted(tally, key=lambda p: -(tally[p]['NATIVE'] /
               max(1, tally[p]['NATIVE'] + tally[p]['ADAPTED'] + tally[p]['GENERIC'])))

fig, ax = plt.subplots(figsize=(10, 6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

# Sequential by intensity of connection to the cuisine, not a rainbow.
STEPS = [('NATIVE', '#1a4f8f', 'native to the cuisine'),
         ('ADAPTED', '#6f9fd8', 'adapted for the park'),
         ('GENERIC', '#c9ccd1', 'generic theme park food')]

for i, p in enumerate(order):
    denom = sum(tally[p][k] for k, _, _ in STEPS)
    left = 0.0
    for key, colour, _ in STEPS:
        w = 100.0 * tally[p][key] / denom
        # 2px surface gap between adjacent segments.
        ax.barh(i, w, left=left, color=colour, height=0.62,
                edgecolor=SURFACE, linewidth=2)
        if w >= 9:
            ax.text(left + w / 2, i, '%.0f%%' % w, va='center', ha='center',
                    fontsize=9.5, color='#ffffff' if key != 'GENERIC' else INK2)
        left += w

ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=11, color=INK2)
ax.set_xlim(0, 100)
ax.set_xticks([])
ax.invert_yaxis()
ax.tick_params(length=0)
for s in ('top', 'right', 'bottom', 'left'):
    ax.spines[s].set_visible(False)
ax.set_title('How much of each pavilion\u2019s menu is actually that '
             'country\u2019s food', fontsize=15, color=INK, pad=16, loc='left')

handles = [plt.Rectangle((0, 0), 1, 1, color=c, label=lab)
           for _, c, lab in STEPS]
ax.legend(handles=handles, loc='lower left', bbox_to_anchor=(0, -0.13),
          frameon=False, ncol=3, fontsize=10, labelcolor=INK2)

plt.tight_layout()
plt.show()

print()
print('The American Adventure is coded on a different basis and is not')
print('comparable here: generic American theme park food is that country\'s')
print('food. It is shown for completeness, not for ranking.')

## What else is in the dataset

- `entertainment.csv` has every act across seven sampled dates, with its set
  length and the quoted source phrase establishing whether it performs in that
  country's own tradition. Good for checking the coding rather than trusting it.
- `architecture.csv` traces 40 pavilion structures to the real buildings they
  reproduce, with a source reliability tier and a flag for the ones sources
  disagree about. Mexico's pyramid is the contested one.
- `shops.csv` is Disney's own shop directory as a snapshot. **No ranking uses
  it.** It is published because the directory assigns zero shops to China,
  which visibly has a large one, and an absent measure should be checkable
  rather than asserted.

## Limits worth respecting if you fork this

Eleven pavilions is eleven data points, so everything here is descriptive and
no correlation computed on it will reach significance at any sensible
threshold.

Buffet and prix fixe venues under-report on item count, because a buffet
publishes as eight stations rather than forty dishes. Biergarten, Akershus,
Takumi-Tei and Monsieur Paul are all affected.

Entertainment is seven sampled dates, not a census. Acts rotate and most take
two days off a week, so a single day would misrepresent any pavilion.

Some of this ages fast. Germany has been under unannounced construction since
July 2026, Morocco's Restaurant Marrakesh has not served food since 2020, and
Wondrous China was announced in 2019 and never opened.

---

Code, methodology notes and a full data dictionary:
[github.com/chadwambles/epcot-world-showcase](https://github.com/chadwambles/epcot-world-showcase)